# Week 9 실습 노트북: CTR 광고 데이터 기반 다목적 최적화

이 노트북은 **하나의 CTR 광고 데이터셋**을 계속 사용하여 다음 내용을 실습합니다.

1. CTR 예측 모델의 성능–비용 트레이드오프 분석
2. Pareto Dominance와 Pareto Front 계산
3. Threshold 조정에 따른 Precision–Recall–Bid Cost 트레이드오프
4. 광고 캠페인 선택 문제에서 CTR, Diversity, Fairness의 다목적 최적화
5. 간단한 NSGA-II 스타일 진화 탐색으로 Pareto 후보 확장

> 모든 데이터는 강의용으로 합성 생성합니다. 외부 데이터 다운로드가 필요 없습니다.

## 0. 라이브러리 준비

Google Colab 또는 로컬 Jupyter에서 실행할 수 있습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. 하나의 CTR 광고 데이터셋 생성

실무 광고 데이터의 전형적인 구성 요소를 반영합니다.

- 사용자 맥락: `age`, `device`, `hour`, `day_of_week`
- 광고 맥락: `ad_category`, `campaign_id`, `bid_price`, `ad_quality`, `popularity`
- 타깃 변수: `clicked` \(0/1\)

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

n = 12000

devices = np.array(['mobile', 'desktop', 'tablet'])
categories = np.array(['fashion', 'electronics', 'finance', 'food', 'game', 'travel'])
campaigns = np.array([f'C{i:02d}' for i in range(1, 21)])

age = np.random.randint(18, 66, size=n)
hour = np.random.randint(0, 24, size=n)
day_of_week = np.random.randint(0, 7, size=n)
device = np.random.choice(devices, size=n, p=[0.62, 0.30, 0.08])
ad_category = np.random.choice(categories, size=n, p=[0.18, 0.20, 0.12, 0.16, 0.18, 0.16])
campaign_id = np.random.choice(campaigns, size=n)
bid_price = np.round(np.random.gamma(shape=2.0, scale=0.55, size=n) + 0.1, 3)
ad_quality = np.clip(np.random.normal(0.58, 0.18, size=n), 0.05, 0.99)
popularity = np.clip(np.random.beta(2, 5, size=n), 0.01, 0.99)

# 클릭 확률을 만드는 숨은 규칙
category_effect = {
    'fashion': 0.15,
    'electronics': 0.35,
    'finance': -0.10,
    'food': 0.10,
    'game': 0.25,
    'travel': 0.05,
}
device_effect = {'mobile': 0.25, 'desktop': 0.05, 'tablet': -0.05}
peak_hour = ((hour >= 19) & (hour <= 23)).astype(float)
weekend = (day_of_week >= 5).astype(float)

logit = (
    -3.2
    + 1.9 * ad_quality
    + 0.32 * bid_price
    + 0.8 * popularity
    + 0.22 * peak_hour
    + 0.12 * weekend
    - 0.006 * np.abs(age - 34)
    + np.vectorize(category_effect.get)(ad_category)
    + np.vectorize(device_effect.get)(device)
)

click_prob = sigmoid(logit)
clicked = np.random.binomial(1, click_prob)

ads = pd.DataFrame({
    'age': age,
    'hour': hour,
    'day_of_week': day_of_week,
    'device': device,
    'ad_category': ad_category,
    'campaign_id': campaign_id,
    'bid_price': bid_price,
    'ad_quality': ad_quality,
    'popularity': popularity,
    'clicked': clicked
})

print('Dataset shape:', ads.shape)
print('Overall CTR:', ads['clicked'].mean().round(4))
ads.head()

## 2. 데이터 분할 및 전처리 파이프라인

동일한 데이터셋을 모든 실습에 사용합니다.

In [ ]:
target = 'clicked'
features = ['age', 'hour', 'day_of_week', 'device', 'ad_category', 'campaign_id', 'bid_price', 'ad_quality', 'popularity']

X = ads[features]
y = ads[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numeric_features = ['age', 'hour', 'day_of_week', 'bid_price', 'ad_quality', 'popularity']
categorical_features = ['device', 'ad_category', 'campaign_id']

preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

print('Train:', X_train.shape, 'Test:', X_test.shape)

## 3. Pareto Front 계산 함수

모든 목적을 **최소화 문제**로 변환한 뒤 Pareto Efficient 후보를 찾습니다.

예: AUC는 클수록 좋으므로 `-AUC`로 변환합니다. 추론 시간과 비용은 작을수록 좋으므로 그대로 사용합니다.

In [ ]:
def is_pareto_efficient(costs):
    """
    costs: shape = (n_points, n_objectives)
    모든 objective는 작을수록 좋다고 가정합니다.
    return: Pareto efficient 여부를 나타내는 boolean mask
    """
    costs = np.asarray(costs)
    n_points = costs.shape[0]
    is_efficient = np.ones(n_points, dtype=bool)

    for i in range(n_points):
        if not is_efficient[i]:
            continue
        # j가 i를 dominate하면 i는 제거
        dominated_by_any = np.any(
            np.all(costs <= costs[i], axis=1) & np.any(costs < costs[i], axis=1)
        )
        if dominated_by_any:
            is_efficient[i] = False
    return is_efficient

# 간단 검증
example_costs = np.array([[1, 5], [2, 3], [3, 2], [4, 4], [5, 1]])
print(is_pareto_efficient(example_costs))

## 4. 여러 CTR 예측 모델 평가: 성능 vs 시간 vs 복잡도

목표는 다음과 같습니다.

- AUC 최대화
- PR-AUC 최대화
- 학습 시간 최소화
- 추론 시간 최소화
- 모델 복잡도 최소화

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'DecisionTree_d3': DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
    'DecisionTree_d8': DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    'RandomForest_50_d5': RandomForestClassifier(n_estimators=50, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1),
    'RandomForest_100_d8': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1),
    'GradientBoosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'KNN_15': KNeighborsClassifier(n_neighbors=15),
}

rows = []
for name, model in models.items():
    pipe = Pipeline([('prep', preprocess), ('model', model)])

    start = time.perf_counter()
    pipe.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    prob = pipe.predict_proba(X_test)[:, 1]
    infer_time = time.perf_counter() - start

    auc = roc_auc_score(y_test, prob)
    pr_auc = average_precision_score(y_test, prob)

    # 교육용 복잡도 지표: 모델별 대략적 상대 복잡도
    if 'Logistic' in name:
        complexity = 1
    elif 'DecisionTree_d3' in name:
        complexity = 3
    elif 'DecisionTree_d8' in name:
        complexity = 8
    elif 'RandomForest_50' in name:
        complexity = 50 * 5
    elif 'RandomForest_100' in name:
        complexity = 100 * 8
    elif 'GradientBoosting' in name:
        complexity = 100 * 3
    elif 'KNN' in name:
        complexity = len(X_train)
    else:
        complexity = 999

    rows.append({
        'model': name,
        'auc': auc,
        'pr_auc': pr_auc,
        'train_time_sec': train_time,
        'infer_time_sec': infer_time,
        'complexity': complexity
    })

model_results = pd.DataFrame(rows).sort_values('auc', ascending=False).reset_index(drop=True)
model_results

In [ ]:
# 목적 변환: AUC와 PR-AUC는 최대화 -> 음수화, 시간과 복잡도는 최소화
costs = model_results[['auc', 'pr_auc', 'infer_time_sec', 'complexity']].copy()
costs['auc'] = -costs['auc']
costs['pr_auc'] = -costs['pr_auc']

pareto_mask = is_pareto_efficient(costs.values)
model_results['pareto'] = pareto_mask
model_results.sort_values(['pareto', 'auc'], ascending=[False, False])

## 5. 모델 Pareto Front 시각화

2차원으로는 `AUC 최대화`와 `추론 시간 최소화`의 트레이드오프를 확인합니다.

In [ ]:
plt.figure(figsize=(9, 6))
for _, row in model_results.iterrows():
    marker = 'o' if row['pareto'] else 'x'
    size = 120 if row['pareto'] else 80
    plt.scatter(row['infer_time_sec'], row['auc'], marker=marker, s=size)
    plt.annotate(row['model'], (row['infer_time_sec'], row['auc']), xytext=(5, 5), textcoords='offset points')

plt.xlabel('Inference Time (sec) - lower is better')
plt.ylabel('AUC - higher is better')
plt.title('CTR Model Trade-off: AUC vs Inference Time')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Threshold 조정: Precision vs Recall vs Bid Cost

동일한 CTR 모델에서 의사결정 임계값을 바꾸면 광고 노출 정책이 달라집니다.

- 낮은 threshold: 더 많은 광고를 노출 → Recall 증가, 비용 증가
- 높은 threshold: 클릭 가능성이 높은 광고만 노출 → Precision 증가, Recall 감소

In [ ]:
best_model_name = model_results.sort_values('auc', ascending=False).iloc[0]['model']
best_pipe = Pipeline([('prep', preprocess), ('model', models[best_model_name])])
best_pipe.fit(X_train, y_train)
prob = best_pipe.predict_proba(X_test)[:, 1]

threshold_rows = []
for th in np.arange(0.05, 0.95, 0.05):
    pred = (prob >= th).astype(int)
    selected = pred == 1
    if selected.sum() == 0:
        avg_bid_cost = 0
    else:
        avg_bid_cost = X_test.loc[selected, 'bid_price'].mean()
    threshold_rows.append({
        'threshold': round(float(th), 2),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'selected_ads': int(selected.sum()),
        'avg_bid_cost': avg_bid_cost
    })

threshold_results = pd.DataFrame(threshold_rows)

# objectives: precision max, recall max, avg_bid_cost min
th_costs = threshold_results[['precision', 'recall', 'avg_bid_cost']].copy()
th_costs['precision'] = -th_costs['precision']
th_costs['recall'] = -th_costs['recall']
threshold_results['pareto'] = is_pareto_efficient(th_costs.values)
threshold_results

In [ ]:
plt.figure(figsize=(9, 6))
for _, row in threshold_results.iterrows():
    marker = 'o' if row['pareto'] else 'x'
    size = 110 if row['pareto'] else 70
    plt.scatter(row['recall'], row['precision'], marker=marker, s=size)
    plt.annotate(str(row['threshold']), (row['recall'], row['precision']), xytext=(4, 4), textcoords='offset points')

plt.xlabel('Recall - higher is better')
plt.ylabel('Precision - higher is better')
plt.title('Threshold Trade-off: Precision vs Recall')
plt.grid(True, alpha=0.3)
plt.show()

## 7. 광고 캠페인 선택: CTR vs Diversity vs Fairness

광고 추천/집행 정책은 단순 CTR만 보면 인기 카테고리나 고입찰 광고에 쏠릴 수 있습니다.
여기서는 캠페인 단위로 집계한 뒤, 다음 목표를 함께 고려합니다.

- 예측 CTR 최대화
- 카테고리 다양성 최대화
- 인기 편향 최소화
- 평균 입찰 비용 최소화

In [ ]:
scored = X_test.copy()
scored['y_true'] = y_test.values
scored['pred_ctr'] = prob

campaign_summary = scored.groupby('campaign_id').agg(
    pred_ctr=('pred_ctr', 'mean'),
    actual_ctr=('y_true', 'mean'),
    avg_bid=('bid_price', 'mean'),
    avg_popularity=('popularity', 'mean'),
    n=('pred_ctr', 'size')
).reset_index()

# 캠페인별 대표 카테고리
main_cat = scored.groupby('campaign_id')['ad_category'].agg(lambda x: x.value_counts().index[0]).reset_index(name='main_category')
campaign_summary = campaign_summary.merge(main_cat, on='campaign_id')
campaign_summary.head()

In [ ]:
def entropy_from_counts(counts):
    counts = np.asarray(counts, dtype=float)
    p = counts / counts.sum()
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

# 여러 정책: pred_ctr에 bid, popularity, diversity penalty를 섞어서 Top-K 캠페인 선택
policy_rows = []
K = 8
for w_ctr in [0.5, 0.7, 0.9, 1.1, 1.3]:
    for w_bid in [0.0, 0.1, 0.2, 0.3]:
        for w_pop_penalty in [0.0, 0.2, 0.4]:
            temp = campaign_summary.copy()
            temp['score'] = (
                w_ctr * temp['pred_ctr']
                - w_bid * temp['avg_bid']
                - w_pop_penalty * temp['avg_popularity']
            )
            selected = temp.sort_values('score', ascending=False).head(K)
            category_counts = selected['main_category'].value_counts().values
            diversity = entropy_from_counts(category_counts)
            fairness = -selected['avg_popularity'].std()  # 값이 클수록 편향이 낮음
            policy_rows.append({
                'policy': f'ctr{w_ctr}_bid{w_bid}_pop{w_pop_penalty}',
                'w_ctr': w_ctr,
                'w_bid': w_bid,
                'w_pop_penalty': w_pop_penalty,
                'mean_pred_ctr': selected['pred_ctr'].mean(),
                'mean_actual_ctr': selected['actual_ctr'].mean(),
                'diversity_entropy': diversity,
                'fairness_neg_pop_std': fairness,
                'mean_bid': selected['avg_bid'].mean(),
                'selected_campaigns': ', '.join(selected['campaign_id'].tolist())
            })

policy_results = pd.DataFrame(policy_rows)

# 목적: pred_ctr max, diversity max, fairness max, mean_bid min
policy_costs = policy_results[['mean_pred_ctr', 'diversity_entropy', 'fairness_neg_pop_std', 'mean_bid']].copy()
policy_costs['mean_pred_ctr'] = -policy_costs['mean_pred_ctr']
policy_costs['diversity_entropy'] = -policy_costs['diversity_entropy']
policy_costs['fairness_neg_pop_std'] = -policy_costs['fairness_neg_pop_std']
policy_results['pareto'] = is_pareto_efficient(policy_costs.values)
policy_results.sort_values(['pareto', 'mean_pred_ctr'], ascending=[False, False]).head(15)

In [ ]:
plt.figure(figsize=(9, 6))
for _, row in policy_results.iterrows():
    marker = 'o' if row['pareto'] else 'x'
    size = 100 if row['pareto'] else 50
    plt.scatter(row['diversity_entropy'], row['mean_pred_ctr'], marker=marker, s=size)

plt.xlabel('Diversity Entropy - higher is better')
plt.ylabel('Mean Predicted CTR - higher is better')
plt.title('Campaign Policy Trade-off: CTR vs Diversity')
plt.grid(True, alpha=0.3)
plt.show()

## 8. NSGA-II 기반 광고 추천 정책 최적화 실습

이 파트에서는 앞에서 만든 **동일한 CTR 광고 데이터셋**과 `campaign_summary`를 그대로 사용합니다.

문제 정의는 다음과 같습니다.

- 하나의 해(solution)는 광고 캠페인 추천 정책의 가중치 벡터입니다.
- 해의 형태: `[w_ctr, w_bid, w_pop]`
- `w_ctr`: 예측 CTR을 얼마나 중요하게 볼 것인가
- `w_bid`: 입찰 비용을 얼마나 강하게 패널티로 줄 것인가
- `w_pop`: 인기 편향을 얼마나 강하게 패널티로 줄 것인가

각 해는 상위 `K=8`개 캠페인을 추천하며, 다음 목적함수를 동시에 최적화합니다.

1. 평균 예측 CTR 최대화
2. 카테고리 다양성 최대화
3. 인기 편향 완화, 즉 popularity 표준편차 최소화
4. 평균 입찰 비용 최소화

강의 목적상 NSGA-II를 외부 라이브러리 없이 직접 구현합니다.

### STEP 1. 광고 추천 후보 생성

초기 population을 생성합니다. 각 후보는 광고 추천 정책을 의미하는 가중치 벡터입니다.

In [ ]:
# STEP 1. 광고 추천 후보 생성

NSGA_RANDOM_STATE = 42
rng = np.random.default_rng(NSGA_RANDOM_STATE)

K = 8
POP_SIZE = 40
N_GENERATIONS = 25

BOUNDS = np.array([
    [0.2, 1.8],  # w_ctr: CTR 중요도
    [0.0, 0.7],  # w_bid: 비용 패널티
    [0.0, 1.0],  # w_pop: 인기 편향 패널티
])


def create_initial_population(pop_size=POP_SIZE, bounds=BOUNDS, rng=rng):
    """광고 추천 정책 후보 population을 생성합니다."""
    low = bounds[:, 0]
    high = bounds[:, 1]
    return rng.uniform(low=low, high=high, size=(pop_size, len(bounds)))


population = create_initial_population()

candidate_df = pd.DataFrame(population, columns=['w_ctr', 'w_bid', 'w_pop'])
candidate_df.head()

### STEP 2. 목적함수 계산

각 후보 정책이 선택한 광고 캠페인 집합을 평가합니다. NSGA-II 내부 계산을 쉽게 하기 위해 모든 목적은 **최대화 방향**으로 맞춥니다.

- `mean_pred_ctr`: 클수록 좋음
- `diversity`: 클수록 좋음
- `neg_popularity_std`: popularity 표준편차에 음수를 붙임 → 클수록 편향이 작음
- `neg_mean_bid`: 평균 입찰 비용에 음수를 붙임 → 클수록 비용이 작음

In [ ]:
# STEP 2. 목적함수 계산

OBJECTIVE_COLUMNS = ['mean_pred_ctr', 'diversity', 'neg_popularity_std', 'neg_mean_bid']


def evaluate_policy_vector(vec, K=K):
    """
    하나의 광고 추천 정책 벡터를 평가합니다.
    vec = [w_ctr, w_bid, w_pop]
    """
    w_ctr, w_bid, w_pop = vec

    temp = campaign_summary.copy()
    temp['score'] = (
        w_ctr * temp['pred_ctr']
        - w_bid * temp['avg_bid']
        - w_pop * temp['avg_popularity']
    )

    selected = temp.sort_values('score', ascending=False).head(K)

    mean_pred_ctr = selected['pred_ctr'].mean()
    diversity = entropy_from_counts(selected['main_category'].value_counts().values)
    neg_popularity_std = -selected['avg_popularity'].std()
    neg_mean_bid = -selected['avg_bid'].mean()

    return np.array([mean_pred_ctr, diversity, neg_popularity_std, neg_mean_bid])


def evaluate_population(population):
    """Population 전체의 목적함수 값을 계산합니다."""
    objectives = np.array([evaluate_policy_vector(ind) for ind in population])
    result = pd.DataFrame(population, columns=['w_ctr', 'w_bid', 'w_pop'])
    result[OBJECTIVE_COLUMNS] = objectives
    return result, objectives


population_df, objectives = evaluate_population(population)
population_df.head()

### STEP 3. Dominance 판별

해 A가 해 B를 지배하려면 다음 두 조건을 모두 만족해야 합니다.

1. 모든 목적에서 A가 B보다 같거나 좋다.
2. 적어도 하나의 목적에서 A가 B보다 더 좋다.

여기서는 모든 목적을 최대화 문제로 맞추었으므로 `>=`, `>` 기준으로 비교합니다.

In [ ]:
# STEP 3. Dominance 판별


def dominates(a, b):
    """최대화 문제에서 a가 b를 지배하는지 판별합니다."""
    return np.all(a >= b) and np.any(a > b)


def fast_non_dominated_sort(objectives):
    """NSGA-II의 핵심인 비지배 정렬을 수행합니다."""
    n = len(objectives)
    dominated_solutions = [[] for _ in range(n)]
    domination_count = np.zeros(n, dtype=int)
    fronts = [[]]

    for p in range(n):
        for q in range(n):
            if p == q:
                continue
            if dominates(objectives[p], objectives[q]):
                dominated_solutions[p].append(q)
            elif dominates(objectives[q], objectives[p]):
                domination_count[p] += 1

        if domination_count[p] == 0:
            fronts[0].append(p)

    i = 0
    while len(fronts[i]) > 0:
        next_front = []
        for p in fronts[i]:
            for q in dominated_solutions[p]:
                domination_count[q] -= 1
                if domination_count[q] == 0:
                    next_front.append(q)
        i += 1
        fronts.append(next_front)

    fronts = fronts[:-1]

    ranks = np.empty(n, dtype=int)
    for rank, front in enumerate(fronts):
        ranks[front] = rank

    return fronts, ranks


fronts, ranks = fast_non_dominated_sort(objectives)
population_df['rank'] = ranks

print('Front 개수:', len(fronts))
print('1번 Pareto Front 후보 수:', len(fronts[0]))
population_df.sort_values(['rank', 'mean_pred_ctr'], ascending=[True, False]).head(10)

### STEP 4. Pareto Front 시각화

첫 번째 Pareto Front는 현재 population에서 서로 지배되지 않는 광고 추천 정책 후보입니다. 2차원 그림에서는 `Diversity`와 `Mean Predicted CTR`의 트레이드오프를 확인합니다.

In [ ]:
# STEP 4. Pareto Front 시각화

plt.figure(figsize=(9, 6))

for _, row in population_df.iterrows():
    is_front0 = row['rank'] == 0
    marker = 'o' if is_front0 else 'x'
    size = 110 if is_front0 else 55
    plt.scatter(row['diversity'], row['mean_pred_ctr'], marker=marker, s=size)

plt.xlabel('Diversity - higher is better')
plt.ylabel('Mean Predicted CTR - higher is better')
plt.title('STEP 4. Initial Population Pareto Front')
plt.grid(True, alpha=0.3)
plt.show()

### STEP 5. Crowding Distance 계산

Crowding Distance는 같은 Pareto Front 안에서 후보들이 얼마나 넓게 퍼져 있는지 측정합니다.

- 값이 클수록 주변에 다른 후보가 적습니다.
- NSGA-II는 좋은 rank를 우선하고, 같은 rank에서는 crowding distance가 큰 후보를 선호합니다.
- 이렇게 하면 CTR만 높은 후보로 쏠리지 않고 다양한 정책 후보를 유지할 수 있습니다.

In [ ]:
# STEP 5. Crowding Distance 계산


def calculate_crowding_distance(objectives, front):
    """하나의 front에 속한 후보들의 crowding distance를 계산합니다."""
    distance = np.zeros(len(front))

    if len(front) == 0:
        return distance
    if len(front) <= 2:
        return np.full(len(front), np.inf)

    front_objectives = objectives[front]
    n_objectives = front_objectives.shape[1]

    for m in range(n_objectives):
        order = np.argsort(front_objectives[:, m])
        distance[order[0]] = np.inf
        distance[order[-1]] = np.inf

        min_value = front_objectives[order[0], m]
        max_value = front_objectives[order[-1], m]
        denom = max_value - min_value

        if denom == 0:
            continue

        for i in range(1, len(front) - 1):
            prev_value = front_objectives[order[i - 1], m]
            next_value = front_objectives[order[i + 1], m]
            distance[order[i]] += (next_value - prev_value) / denom

    return distance


crowding = np.zeros(len(population))
for front in fronts:
    distances = calculate_crowding_distance(objectives, front)
    for local_idx, global_idx in enumerate(front):
        crowding[global_idx] = distances[local_idx]

population_df['crowding_distance'] = crowding
population_df.sort_values(['rank', 'crowding_distance'], ascending=[True, False]).head(10)

### STEP 6. Selection

Tournament Selection을 사용합니다.

두 후보를 무작위로 뽑고, 다음 기준으로 부모를 선택합니다.

1. rank가 낮은 후보를 선택합니다.
2. rank가 같으면 crowding distance가 큰 후보를 선택합니다.

In [ ]:
# STEP 6. Selection


def tournament_selection(population, ranks, crowding, rng=rng):
    """NSGA-II 방식의 binary tournament selection입니다."""
    i, j = rng.choice(len(population), size=2, replace=False)

    if ranks[i] < ranks[j]:
        return population[i]
    if ranks[j] < ranks[i]:
        return population[j]

    if crowding[i] >= crowding[j]:
        return population[i]
    return population[j]


selected_parents = np.array([
    tournament_selection(population, ranks, crowding)
    for _ in range(8)
])

pd.DataFrame(selected_parents, columns=['w_ctr', 'w_bid', 'w_pop'])

### STEP 7. Mutation/Crossover

선택된 부모로부터 자식 후보를 만듭니다.

- Crossover: 두 부모의 가중치를 섞어 새로운 정책 생성
- Mutation: 일부 가중치에 작은 노이즈를 추가하여 탐색 범위 확장
- Clip: 가중치가 허용 범위를 벗어나지 않도록 보정

In [ ]:
# STEP 7. Mutation/Crossover


def crossover(parent1, parent2, rng=rng):
    """두 부모의 선형 결합으로 자식 1개를 생성합니다."""
    alpha = rng.uniform(0, 1)
    return alpha * parent1 + (1 - alpha) * parent2


def mutate(child, mutation_rate=0.3, mutation_scale=np.array([0.12, 0.05, 0.08]), bounds=BOUNDS, rng=rng):
    """일정 확률로 각 가중치에 Gaussian noise를 추가합니다."""
    mutated = child.copy()
    for i in range(len(mutated)):
        if rng.random() < mutation_rate:
            mutated[i] += rng.normal(0, mutation_scale[i])

    return np.clip(mutated, bounds[:, 0], bounds[:, 1])


def make_offspring(population, ranks, crowding, pop_size=POP_SIZE, rng=rng):
    """Selection, crossover, mutation을 이용해 offspring population을 생성합니다."""
    offspring = []
    while len(offspring) < pop_size:
        parent1 = tournament_selection(population, ranks, crowding, rng)
        parent2 = tournament_selection(population, ranks, crowding, rng)
        child = crossover(parent1, parent2, rng)
        child = mutate(child, rng=rng)
        offspring.append(child)
    return np.array(offspring)


offspring = make_offspring(population, ranks, crowding)

pd.DataFrame(offspring[:10], columns=['w_ctr', 'w_bid', 'w_pop'])

### STEP 8. 세대 반복

NSGA-II는 부모 population과 자식 population을 합친 뒤, 비지배 정렬과 crowding distance 기준으로 다음 세대를 구성합니다.

즉, 매 세대마다 다음 과정을 반복합니다.

1. 현재 population 평가
2. rank와 crowding distance 계산
3. selection, crossover, mutation으로 offspring 생성
4. 부모와 offspring을 합침
5. 좋은 rank와 큰 crowding distance 기준으로 다음 population 선택

In [ ]:
# STEP 8. 세대 반복


def assign_rank_and_crowding(population):
    """Population의 목적함수, rank, crowding distance를 한 번에 계산합니다."""
    df, objectives = evaluate_population(population)
    fronts, ranks = fast_non_dominated_sort(objectives)

    crowding = np.zeros(len(population))
    for front in fronts:
        distances = calculate_crowding_distance(objectives, front)
        for local_idx, global_idx in enumerate(front):
            crowding[global_idx] = distances[local_idx]

    df['rank'] = ranks
    df['crowding_distance'] = crowding
    return df, objectives, fronts, ranks, crowding


def environmental_selection(combined_population, pop_size=POP_SIZE):
    """부모+자식 중 다음 세대에 남길 후보를 선택합니다."""
    df, objectives, fronts, ranks, crowding = assign_rank_and_crowding(combined_population)

    selected_indices = []
    for front in fronts:
        if len(selected_indices) + len(front) <= pop_size:
            selected_indices.extend(front)
        else:
            front_distances = calculate_crowding_distance(objectives, front)
            sorted_front = [idx for _, idx in sorted(zip(front_distances, front), reverse=True)]
            remaining = pop_size - len(selected_indices)
            selected_indices.extend(sorted_front[:remaining])
            break

    return combined_population[selected_indices]


population = create_initial_population()
history = []
pareto_snapshots = {}

for generation in range(N_GENERATIONS):
    pop_df, objectives, fronts, ranks, crowding = assign_rank_and_crowding(population)
    pop_df['generation'] = generation
    pop_df['is_pareto_front'] = pop_df['rank'] == 0
    history.append(pop_df)

    if generation in [0, 4, 9, 14, 19, N_GENERATIONS - 1]:
        pareto_snapshots[generation] = pop_df[pop_df['rank'] == 0].copy()

    offspring = make_offspring(population, ranks, crowding)
    combined_population = np.vstack([population, offspring])
    population = environmental_selection(combined_population)

nsga_history = pd.concat(history, ignore_index=True)
final_population_df = nsga_history[nsga_history['generation'] == N_GENERATIONS - 1].copy()
final_pareto_df = final_population_df[final_population_df['rank'] == 0].copy()

print('최종 세대 Pareto 후보 수:', len(final_pareto_df))
final_pareto_df.sort_values(['mean_pred_ctr', 'diversity'], ascending=False).head(10)

### STEP 9. Pareto Front 진화 관찰

세대가 반복될수록 후보 정책들이 더 나은 trade-off 영역으로 이동하는지 확인합니다.

아래 그림에서는 여러 세대의 Pareto Front를 함께 표시합니다.

In [ ]:
# STEP 9. Pareto Front 진화 관찰

plt.figure(figsize=(10, 7))

for generation, snap in pareto_snapshots.items():
    plt.scatter(
        snap['diversity'],
        snap['mean_pred_ctr'],
        s=80,
        alpha=0.65,
        label=f'Gen {generation}'
    )

plt.xlabel('Diversity - higher is better')
plt.ylabel('Mean Predicted CTR - higher is better')
plt.title('STEP 9. Evolution of Pareto Fronts')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 최종 Pareto 후보 상세 확인

final_view = final_pareto_df[[
    'generation', 'w_ctr', 'w_bid', 'w_pop',
    'mean_pred_ctr', 'diversity', 'neg_popularity_std', 'neg_mean_bid',
    'crowding_distance'
]].copy()

final_view['popularity_std'] = -final_view['neg_popularity_std']
final_view['mean_bid'] = -final_view['neg_mean_bid']

final_view = final_view.drop(columns=['neg_popularity_std', 'neg_mean_bid'])
final_view.sort_values(['mean_pred_ctr', 'diversity'], ascending=False).head(15)

#### 실습 해석 포인트

- `w_ctr`가 큰 후보는 평균 CTR을 높이는 방향으로 광고를 추천합니다.
- `w_bid`가 큰 후보는 고비용 광고를 피하려는 경향이 강합니다.
- `w_pop`가 큰 후보는 popularity가 높은 캠페인에 과도하게 쏠리는 현상을 줄입니다.
- Pareto Front에 있는 후보들은 단일 최적해가 아니라, 의사결정자가 선택할 수 있는 **정책 대안 집합**입니다.
- 광고 비즈니스에서는 CTR만 높은 정책보다 비용, 다양성, 공정성을 함께 고려한 Pareto 후보가 더 실무적인 의사결정 근거가 됩니다.

## 9. 과제용 분석 질문

아래 질문에 답하면 Week 9 과제 리포트 구조로 사용할 수 있습니다.

1. 모델 선택 실험에서 Pareto 모델은 어떤 공통점을 가지는가?
2. AUC가 가장 높은 모델이 항상 최선인가? 추론 시간이 중요한 서비스에서는 어떤 모델을 선택할 것인가?
3. Threshold를 높이면 Precision, Recall, 광고 노출 수, 평균 입찰 비용은 어떻게 변하는가?
4. 캠페인 정책에서 CTR만 극대화하면 Diversity와 Fairness는 어떻게 변하는가?
5. 최종 의사결정자가 `매출 중심`, `사용자 경험 중심`, `비용 절감 중심`일 때 각각 어떤 Pareto 후보를 추천할 것인가?